*Notebook 05 in `notebooks/arcs/`. Parameters (pre-registered): `analysis/arcs/config.yaml`; decisions and run notes: `analysis/arcs/LOG.md`. The static page `site/arcs/index.html` is built from the same saved outputs.*

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]   # notebooks/arcs -> repo root
sys.path.insert(0, str(REPO))

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

from analysis.arcs import s05_arcs as step

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Step 5 · Arcs

**What we do.** For each script we take its scene scores in story order: the "both" layer, one score per
scene on the timeline. Then:
1. We put them on a common clock of **100 points of story time**. Scene *i* of *n* covers
   [*i*/*n*, (*i*+1)/*n*) of the story. Each grid point takes the mean of the scenes it overlaps, weighted by
   the overlap. This works the same for a script with 40 scenes as for one with 400; nothing is dropped or
   interpolated.
2. We **smooth** with a centred moving average over 11 points (about a tenth of the story), cut short at the
   two ends.
3. We summarise each arc with the two pre-registered **primary descriptors**:
   - **D1, volatility**: the standard deviation of the smoothed arc. How much the tone swings.
   - **D2, ending minus start**: the mean of the last 20% minus the mean of the first 20% of the unsmoothed
     arc. Does the story end lighter or darker than it began.

   Three exploratory descriptors come alongside (mean level, position of the low point, number of swings).

**Two checks.**
- **Proxies.** Are the descriptors just script size in disguise? A scene made of two sentences gives a noisy
  mean, so scripts of short scenes could look volatile for no narrative reason.
- **Reliability.** Each script is split into odd and even scenes, and every descriptor is computed on each
  half. If the halves disagree, scene noise dominates.

The instrument here is VADER. The arcs are rebuilt with the contextual model once the HPC job returns, and the
fallback rule decides which is primary. No outcome is used in this step.

In [ ]:
res = step.run()   # recomputes this step and saves its outputs

/Users/gyula/Documents/Projects/code/IMDb-movie-success-arcs/analysis/arcs/s05_arcs.py:35: RuntimeWarning: invalid value encountered in divide
  a = np.arange(n + 1) / n


**Figure.** Mean arc over the primary sample. Grey: smoothed. Blue: resampled, unsmoothed. Bands: ±1.96 standard errors across films.

In [ ]:
step.figures(res)["mean"].show()

**Figure.** Mean smoothed arc per genre bin.

In [ ]:
step.figures(res)["genres"].show()

**Table.** Median descriptors per genre bin (primary sample).

In [ ]:
display(step.tables(res)["genres"])

,primary_genre,films,D1,D2,mean_level
0,Action,66,0.0368,-0.0277,-0.0195
1,Adventure,113,0.0380,-0.0163,0.0027
2,Comedy,175,0.0359,-0.0119,0.0317
3,Drama,203,0.0376,-0.0094,0.0143
4,Horror,98,0.0398,-0.0320,-0.0177
5,Other,4,0.0402,-0.0475,-0.0285
6,Sci-Fi,71,0.0394,-0.0244,-0.0109
7,Thriller,104,0.0361,-0.0211,-0.0029


**Figure.** Six arcs drawn with the config seed: numbers only, no script text.

In [ ]:
step.figures(res)["examples"].show()

**Table.** The six example scripts and their descriptors.

In [ ]:
display(step.tables(res)["examples"])

,movie_name,primary_genre,scenes,D1,D2,mean_level
6,127 Hours_2010,Drama,231,0.0293,0.0147,0.0197
111,Barbarian_2022,Horror,64,0.0381,-0.0897,-0.0147
373,Hanna_2011,Adventure,207,0.0365,-0.0535,-0.0126
404,House of 1000 Corpses_2003,Horror,129,0.0396,-0.0212,-0.0140
580,Next_2007,Sci-Fi,75,0.0204,-0.0141,0.0146
794,The Accountant_2016,Action,159,0.0467,0.0064,-0.0230


**Figure.** Distributions of D1 and D2 in the primary sample.

In [ ]:
step.figures(res)["descriptors"].show()

**Table.** Descriptor quantiles in the primary sample.

In [ ]:
display(step.tables(res)["distribution"])

,descriptor (primary 834),count,mean,std,min,10%,25%,50%,75%,90%,max
0,D1,834.0,0.0392,0.0131,0.0143,0.0245,0.0300,0.0376,0.0458,0.0561,0.0978
1,D2,834.0,-0.0175,0.0484,-0.2426,-0.0747,-0.0506,-0.0181,0.0127,0.0409,0.1246
2,mean_level,834.0,0.0046,0.0349,-0.1126,-0.0368,-0.0172,0.0035,0.0274,0.0466,0.1887
3,min_position,834.0,0.5741,0.3024,0.0050,0.1150,0.3250,0.6350,0.8450,0.9250,0.9950
4,sign_changes,834.0,9.3549,3.6281,1.0000,5.0000,7.0000,9.0000,12.0000,14.0000,23.0000


**Table.** Spearman correlation of each descriptor with script size (primary sample).

In [ ]:
display(step.tables(res)["proxies"])

against,descriptor,mean sentences per scene,scenes on the timeline,sentences
0,D1,-0.014,-0.096,-0.224
1,D2,0.075,-0.058,0.019
2,mean_level,0.126,-0.141,-0.024


**Figure.** D1 against mean scene length: the scene-noise effect that the scene-length control addresses (the Spearman correlation is in the proxies table).

In [ ]:
step.figures(res)["noise"].show()

**Table.** Split-half reliability: odd against even scenes, Spearman correlation across films and its Spearman-Brown correction to full length.

In [ ]:
display(step.tables(res)["reliability"])

,descriptor,split-half Spearman,Spearman-Brown
0,D1,0.363,0.532
1,D2,0.293,0.453
2,mean_level,0.791,0.883


**Figure.** D1 computed on odd scenes against D1 on even scenes.

In [ ]:
step.figures(res)["split_half"].show()

**Plain-language reading** (generated from the saved numbers):

In [ ]:
display(Markdown(step.reading(res)))

Every one of the 1,087 scripts has an arc (834 in the primary sample). The median primary script has 133 scenes on its timeline, about 1.3 scenes per grid point. The average smoothed arc stays close to zero: +0.015 over the first 20%, -0.004 over the last 20%, and between -0.010 and +0.020 overall. Individual arcs swing much more than the average, which mostly cancels out (median D1 0.038, median D2 -0.018, 66% of films end lower than they start). The scores are VADER and NOT YET VALIDATED. **Proxies.** The strongest size correlate of D1 is sentences (ρ = -0.22); of D2, mean sentences per scene (ρ = +0.07); of mean level, scenes on the timeline (ρ = -0.14). All are weak (|ρ| < 0.3), so the descriptors are not script size in disguise. The size controls of Step 7 still apply. **Reliability.** Split-half (odd vs even scenes, Spearman-Brown corrected): D1 0.53, D2 0.45, mean level 0.88. A reliability of r means roughly a share 1 − r of the between-film variance is scene-level noise: 47% for D1 and 55% for D2, against 12% for the mean level. That noise caps how strongly a descriptor can correlate with anything, profit included. Step 6 checks whether the order of scenes carries any structure at all.